In [ ]:
from google.colab import files
files.upload()

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:02<00:00, 149MB/s]



In [ ]:
!unzip utkface-new.zip

In [5]:
import os
import pandas as pd
data_dir = 'utkface_aligned_cropped/UTKFace'
if not os.path.exists(data_dir):
    data_dir = 'UTKFace'

image_paths = []
ages = []
genders = []
races = []

files = os.listdir(data_dir)
print(f"Tìm thấy {len(files)} tệp trong {data_dir}")

for filename in files:
    if filename.endswith(".jpg"):
        parts = filename.split('_')
        if len(parts) >= 4:
            image_paths.append(os.path.join(data_dir, filename))
            ages.append(int(parts[0]))
            genders.append(int(parts[1]))
            races.append(int(parts[2]))

df = pd.DataFrame({
    'image_path': image_paths,
    'age': ages,
    'gender': genders,
    'race': races
})

display(df.head())
print(f"Tổng số mẫu đã load: {len(df)}")

Tìm thấy 23708 tệp trong utkface_aligned_cropped/UTKFace


,image_path,age,gender,race
0,utkface_aligned_cropped/UTKFace/6_0_4_20161221...,6,0,4
1,utkface_aligned_cropped/UTKFace/50_0_0_2017010...,50,0,0
2,utkface_aligned_cropped/UTKFace/47_1_0_2017010...,47,1,0
3,utkface_aligned_cropped/UTKFace/55_0_3_2017011...,55,0,3
4,utkface_aligned_cropped/UTKFace/1_1_0_20170109...,1,1,0


Tổng số mẫu đã load: 23705


In [6]:
import cv2
import numpy as np
from tqdm import tqdm

def preprocess_images(image_paths, target_size=(64, 64)):
    X = []
    for img_path in tqdm(image_paths, desc="processing"):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, target_size)
        img = img.flatten() / 255.0
        X.append(img)
    return np.array(X)

X = preprocess_images(df['image_path'].values)

print(f"\nShape : {X.shape}")


processing: 100%|██████████| 23705/23705 [00:10<00:00, 2316.48it/s]



Shape : (23705, 4096)


In [7]:
X[0:2]

array([[0.95686275, 0.89411765, 0.45098039, ..., 0.04313725, 0.03921569,
        0.03529412],
       [0.1254902 , 0.0627451 , 0.0627451 , ..., 0.71764706, 0.71764706,
        0.71764706]])

In [8]:
from sklearn.model_selection import train_test_split

y_age = df['age'].values
y_gender = df['gender'].values
y_race = df['race'].values

In [9]:
y_gender[0:10]

array([0, 0, 1, 0, 1, 0, 1, 0, 1, 0])

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y_gender, test_size=0.2, random_state=42)
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Male', 'Female']))

Accuracy: 83.80%

Classification Report:
              precision    recall  f1-score   support

        Male       0.85      0.85      0.85      2501
      Female       0.83      0.83      0.83      2240

    accuracy                           0.84      4741
   macro avg       0.84      0.84      0.84      4741
weighted avg       0.84      0.84      0.84      4741



In [11]:
from sklearn.linear_model import Perceptron
from sklearn.neighbors import KNeighborsClassifier

results = {}
results['Logistic Regression'] = accuracy

# Weighted Logistic Regression
log_reg_weighted = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg_weighted.fit(X_train, y_train)
y_pred_w = log_reg_weighted.predict(X_test)
results['Weighted Logistic Regression'] = accuracy_score(y_test, y_pred_w)

# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
results['KNN'] = accuracy_score(y_test, y_pred_knn)

# Perceptron
percept = Perceptron(max_iter=1000, tol=1e-3)
percept.fit(X_train, y_train)
y_pred_percept = percept.predict(X_test)
results['Perceptron'] = accuracy_score(y_test, y_pred_percept)

In [12]:
comparison_df = pd.DataFrame(list(results.items()), columns=['Algorithm', 'Accuracy'])
display(comparison_df.sort_values(by='Accuracy', ascending=False))

,Algorithm,Accuracy
1,Weighted Logistic Regression,0.839485
0,Logistic Regression,0.838009
3,Perceptron,0.799198
2,KNN,0.727695
